In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, StringType
import pyspark.sql.functions as sf
from delta.tables import DeltaTable
from utils.data_models import make_df_1m_price_new

In [ ]:
#catalog = spark.conf.get("catalog")
catalog = "runescape_dev"

In [ ]:
# create path and schema for table
path = f"/Volumes/{catalog}/00_landing/data_sources/latest_prices/"

In [ ]:
# read in json data into df_raw
df_raw = spark.read.json(path, multiLine =True)

# transform data into 1m price df 
df_1m_prices = make_df_1m_price_new(spark, df_raw)

In [ ]:
# Insert df_1m_prices into '{catalog}.01_bronze.latest_prices_raw' table

targetDF = DeltaTable.forName(spark, f"{catalog}.01_bronze.latest_prices_raw")
dfUpdates = df_1m_prices

# TODO make sure this merge condition is good.
targetDF.alias("t") .\
  merge(
    source = dfUpdates.alias("s"),
    condition = "t.time = s.time AND t.id = s.id AND t.highorlow = t.highorlow") .\
  whenNotMatchedInsertAll() .\
  execute()